# Waveguide-to-fiber grating coupler

Launch a guided mode and calculate coupling into a 10° Gaussian fiber mode from saved complex fields.

In [ ]:
from math import radians, sin

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import SymLogNorm

from gsim import fdtd

## Build the apodized grating

In [ ]:
pdk = gf.gpdk.PDK
layer_stack = pdk.layer_stack.model_copy(deep=True)
layer_stack.layers["substrate"].background = True
pdk = pdk.model_copy(update={"name": "generic_soi_grating", "layer_stack": layer_stack})
pdk.activate()
# Apodize the fill fraction and chirp the period across 24 teeth.
position = np.linspace(0.0, 1.0, 24)
fill_fraction = 0.55 + (0.91 - 0.55) * (1 - position) ** 1.8
index_model = 2.732 + 0.35 * (fill_fraction - 0.343 / 0.667)
in_plane_index = 1.443 * sin(radians(10.0))
periods_um = 0.6255 * (2.732 - in_plane_index) / (index_model - in_plane_index)
periods_um *= 1 - 0.01 * (position - 0.5)
widths_um = np.round(periods_um * fill_fraction, 3)
gaps_um = np.round(periods_um - widths_um, 3)

component = gf.Component()
grating = component.add_ref(
    gf.components.grating_coupler_elliptical_arbitrary(
        gaps=tuple(gaps_um),
        widths=tuple(widths_um),
        fiber_angle=10.0,
        taper_length=14.0,
        taper_angle=40.0,
        wavelength=1.554,
        layer_slab="SLAB150",
        taper_to_slab_offset=-3.0,
    )
)
straight = component.add_ref(gf.components.straight(length=3.0))
straight.connect("o2", grating.ports["o1"])
component.add_port("o1", port=straight.ports["o1"])
silicon_layer = pdk.layer_stack.layers["core"]
silicon_middle_um = silicon_layer.zmin + silicon_layer.thickness / 2
silicon_top_um = silicon_layer.zmin + silicon_layer.thickness

## Record the fields

Capture tangential E/H above the grating for fiber overlap, guided input power, and Ey along the vertical grating centerline.

In [ ]:
fiber_center_x_um, fiber_y_um = 20.5, 0.0
fiber_size_um = (26.0, 26.0, 0.0)
half_aperture_um = 26.0 / 2

In [ ]:
simulation = fdtd.Simulation(pdk=pdk)
simulation.materials(background="sio2", overrides={"si": 3.47, "sio2": 1.443})
simulation.geometry(component, mesh_size_nm=500, geometry_tolerance_nm=20.0)
simulation.source(
    port="o1", wavelength_um=1.554, wavelength_span_um=0.1, num_wavelengths=101
)
simulation.monitors.add_plane(
    name="radiation",
    center_um=(fiber_center_x_um, fiber_y_um, 1.05),
    size_um=fiber_size_um,
    normal="+z",
    flux=True,
    fields=fdtd.Fields(components=["ex", "ey", "hx", "hy"]),
)
simulation.monitors.add_plane(
    name="input_power",
    center_um=(
        component.xmin + 1.5,
        float(component.ports["o1"].center[1]),
        silicon_middle_um,
    ),
    size_um=(
        0.0,
        float(component.ports["o1"].width) + 2 * 0.5,
        silicon_layer.thickness + 2 * 0.5,
    ),
    normal="+x",
    flux=True,
)
grating_x_bounds = (component.xmin, component.xmax + 0.5)
side_z_bounds = (-4.0, 1.55)
simulation.monitors.add_plane(
    name="grating_side",
    center_um=(
        sum(grating_x_bounds) / 2,
        fiber_y_um,
        sum(side_z_bounds) / 2,
    ),
    size_um=(
        grating_x_bounds[1] - grating_x_bounds[0],
        0.0,
        side_z_bounds[1] - side_z_bounds[0],
    ),
    normal="+y",
    flux=False,
    fields=fdtd.Fields(components=["ey"], wavelengths_um=[1.55]),
)
simulation.domain(
    padding=0.5,
    x_bounds=(
        component.xmin,
        max(component.xmax, fiber_center_x_um + half_aperture_um) + 0.5,
    ),
    y_bounds=(
        min(component.ymin, fiber_y_um - half_aperture_um) - 0.5,
        max(component.ymax, fiber_y_um + half_aperture_um) + 0.5,
    ),
    z_bounds=side_z_bounds,
    pml_cells=16,
    clip_geometry=True,
)
simulation.solver(cell_size_nm=50, energy_decay_fraction=0.0004);

In [ ]:
simulation.plot_3d()

### Check the cross-section

Check source and monitor placement; the Z scale is stretched to resolve the thin silicon, and hatched regions mark PML.

In [ ]:
simulation.plot_2d_static(axis="y", position_um=fiber_y_um, legend=False, aspect="auto")

## Run and calculate fiber coupling

The model retains the 3 µm buried oxide and silicon substrate; check mesh and domain convergence before relying on efficiency values.

For the uniform, lossless single-mode input lead, `P_incident = P_net / (1 - |S11|²)`. The small helper below projects the +Z radiation plane onto a Y-polarized Gaussian, including diffraction and phase curvature. Its complex amplitude retains the simulation phase reference; CE is `|amplitude|² / P_incident`.

In [ ]:
result = simulation.run(check_cache=True)
assert result.convergence["converged"], "Increase run duration to reach convergence."

In [ ]:
def fiber_coupling(
    plane,
    wavelength_um,
    incident_power,
    focus_um,
    tilt_deg=10.0,
    waist_um=5.0,
    refractive_index=1.443,
):
    """Complex overlap and CE for a Y-polarized Gaussian on the +Z XY plane."""
    ey = plane.field("ey", wavelength_um=wavelength_um)
    hx = plane.field("hx", wavelength_um=wavelength_um)
    if not all(
        np.isclose(f.wavelength_um, wavelength_um, rtol=1e-10, atol=0) for f in (ey, hx)
    ):
        raise ValueError("Requested wavelength was not captured")
    coordinates = ey.coordinates
    x, y = np.meshgrid(coordinates.u_um - focus_um[0], coordinates.v_um - focus_um[1])
    z = coordinates.normal_position_um - focus_um[2]
    angle = np.deg2rad(tilt_deg)
    axial = np.sin(angle) * x + np.cos(angle) * z
    radius_squared = np.maximum(x * x + y * y + z * z - axial * axial, 0)
    k = 2 * np.pi * refractive_index / wavelength_um
    rayleigh = k * waist_um**2 / 2
    # The complex beam factor includes diffraction, curvature and Gouy phase.
    q = 1 - 1j * axial / rayleigh
    gaussian = np.exp(-1j * k * axial - radius_squared / (waist_um**2 * q)) / q
    density = gaussian.conj() * (
        refractive_index * np.cos(angle) * ey.load() - hx.load()
    )
    length_um = ey.metadata["solver_time_unit_seconds"] * 299792458.0 * 1e6
    integral = np.trapezoid(
        np.trapezoid(density, x=coordinates.u_um / length_um, axis=1),
        x=coordinates.v_um / length_um,
    )
    mode_power = np.pi * refractive_index * (waist_um / length_um) ** 2 / 4
    amplitude = integral / (4 * np.sqrt(mode_power))
    return amplitude, abs(amplitude) ** 2 / incident_power

In [ ]:
source_port = result.excited_port
reflection = result.s_parameters[(source_port, source_port)].values
incident_power = np.full_like(result.wavelength_um, np.nan)
incident_power[result.valid] = result.monitors["input_power"].flux[result.valid] / (
    1 - np.abs(reflection[result.valid]) ** 2
)
assert np.all(
    np.isfinite(incident_power[result.valid]) & (incident_power[result.valid] > 0)
)
fiber_focus_um = (
    fiber_center_x_um - 0.8,
    fiber_y_um,
    silicon_top_um,
)
fiber_amplitude = np.full_like(result.wavelength_um, np.nan, dtype=complex)
efficiency = np.full_like(result.wavelength_um, np.nan)
for index in np.flatnonzero(result.valid):
    fiber_amplitude[index], efficiency[index] = fiber_coupling(
        result.monitors["radiation"],
        result.wavelength_um[index],
        incident_power[index],
        fiber_focus_um,
    )
coupling = fdtd.CouplingEfficiencyResult(
    label="Fiber",
    wavelength_um=result.wavelength_um,
    efficiency=efficiency,
    valid=result.valid,
)
coupling.plot_plotly(db=False)

## Grating fields

View Re(Ey) at 1.550 µm along the grating centerline at y = 0.00 µm. The symmetric logarithmic color scale shows the signed field, with material boundaries drawn over it.

In [ ]:
sample = result.monitors["grating_side"].field("ey", wavelength_um=1.55)
if not np.isclose(sample.wavelength_um, 1.55, rtol=1e-10, atol=0):
    raise ValueError("Ey was not captured at 1.550 µm")
coordinates = sample.coordinates
# The +Y plane stores (Z, X); transpose it to put X on the horizontal axis.
electric = sample.load().T
peak_field = np.max(np.abs(electric))
field_values = electric.real / peak_field
x, z = coordinates.v_um, coordinates.u_um
field_norm = SymLogNorm(linthresh=0.03, vmin=-1, vmax=1)
figure, axis = plt.subplots(figsize=(8, 3), layout="constrained")
image = axis.pcolormesh(
    x, z, field_values, shading="auto", cmap="RdBu_r", norm=field_norm, rasterized=True
)
axis.set(
    title=f"Re(Ey) at λ = {sample.wavelength_um:.3f} µm",
    xlabel="X (µm)",
    ylabel="Z (µm)",
    xlim=(x[0], x[-1]),
    ylim=(z[0], z[-1]),
)
colorbar = figure.colorbar(image, ax=axis, label="Normalized Re(Ey)", shrink=0.8)
colorbar.set_ticks(
    [
        tick
        for tick in colorbar.get_ticks()
        if tick == 0 or field_norm.linthresh <= abs(tick) <= 1
    ]
)
simulation.plot_2d_static(
    axis="y", position_um=coordinates.normal_position_um, ax=axis, outline_only=True
)
plt.show()

## Fiber alignment tolerance

Sweep position and tilt around the receiver used for the coupling spectrum; each curve varies one parameter at 1.550 µm. Dotted lines mark that receiver setting.

In [ ]:
reference_index = int(np.argmin(abs(result.wavelength_um - 1.55)))


def alignment_ce_percent(x_shift_um=0.0, y_shift_um=0.0, tilt_deg=10.0):
    """Vary one receiver setting at the operating wavelength."""
    focus = (
        fiber_focus_um[0] + x_shift_um,
        fiber_focus_um[1] + y_shift_um,
        fiber_focus_um[2],
    )
    _, ce = fiber_coupling(
        result.monitors["radiation"],
        1.55,
        incident_power[reference_index],
        focus,
        tilt_deg=tilt_deg,
    )
    return 100 * ce


displacements_um = np.linspace(-5.0, 5.0, 41)
tilt_offsets_deg = np.linspace(-5.0, 5.0, 81)
tilt_angles_deg = 10.0 + tilt_offsets_deg
x_ce_percent = [alignment_ce_percent(x_shift_um=shift) for shift in displacements_um]
y_ce_percent = [alignment_ce_percent(y_shift_um=shift) for shift in displacements_um]
tilt_ce_percent = [alignment_ce_percent(tilt_deg=angle) for angle in tilt_angles_deg]
figure, axes = plt.subplots(1, 3, sharey=True, layout="constrained", figsize=(8, 3))
for axis, values, efficiency, label, nominal in zip(
    axes,
    (displacements_um, displacements_um, tilt_offsets_deg),
    (x_ce_percent, y_ce_percent, tilt_ce_percent),
    ("X displacement (µm)", "Y displacement (µm)", "Fiber tilt offset (°)"),
    (0.0, 0.0, 0.0),
    strict=True,
):
    axis.plot(values, efficiency)
    axis.axvline(nominal, color="0.5", linestyle=":", linewidth=1)
    axis.set(xlabel=label, xlim=(values[0], values[-1]))
    axis.set_xticks(np.linspace(values[0], values[-1], 5))
    axis.grid(alpha=0.2)
axes[0].set_ylim(0, 1.05 * max([*x_ce_percent, *y_ce_percent, *tilt_ce_percent]))
axes[0].set_ylabel("Coupling efficiency (%)")
figure.suptitle("Fiber alignment at λ = 1.550 µm")
plt.show()